In [1]:
# Import libraries
import pandas as pd
import numpy as np 

In [ ]:
# Read the CSV/Excel files into DataFrames
weather_2026 = pd.read_csv(
    r"C:\Users\zychl\Desktop\Data Engineering\Python\weather_data_processing\00 raw\2026.csv.gz",
    header=None
)

weather_2025 = pd.read_csv(
    r"C:\Users\zychl\Desktop\Data Engineering\Python\weather_data_processing\00 raw\2025.csv.gz",
    header=None
)

weather_2024 = pd.read_csv(
    r"C:\Users\zychl\Desktop\Data Engineering\Python\weather_data_processing\00 raw\2024.csv.gz",
    header=None
)

weather_2023 = pd.read_csv(
    r"C:\Users\zychl\Desktop\Data Engineering\Python\weather_data_processing\00 raw\2023.csv.gz",
    header=None
)



In [ ]:
stations_raw = pd.read_csv(
    r"C:\Users\zychl\Desktop\Data Engineering\Python\weather_data_processing\00 raw\ghcnd-stations.csv",
    # Use on_bad_lines='skip' because some rows in the downloaded station file
    # contain an unexpected number of fields, which would stop parsing.  
    on_bad_lines="skip",
    header=None
)

In [13]:
weather_2023.head()

,0,1,2,3,4,5,6,7
0,ASN00009515,20230101,PRCP,0,NaN,NaN,a,NaN
1,ASN00009518,20230101,TMAX,223,NaN,NaN,a,NaN
2,ASN00009518,20230101,TMIN,166,NaN,NaN,a,NaN
3,ASN00009518,20230101,PRCP,0,NaN,NaN,a,NaN
4,ASN00009518,20230101,TAVG,182,H,NaN,S,NaN


In [11]:
stations_raw.head()

,0,1,2,3,4,5,6,7,8
0,ACW00011604,17.1167,-61.7833,10.1,,ST JOHNS COOLIDGE FLD,,,
1,ACW00011647,17.1333,-61.7833,19.2,,ST JOHNS,,,
2,AE000041196,25.3330,55.5170,34.0,,SHARJAH INTER. AIRP,GSN,,41196
3,AEM00041194,25.2550,55.3640,10.4,,DUBAI INTL,,,41194
4,AEM00041217,24.4330,54.6510,26.8,,ABU DHABI INTL,,,41217


In [ ]:
weather_2023 = raw_2023.copy()
weather_2024 = raw_2024.copy() 

In [ ]:
# Clean 2023 and 2024 dfs 
weather_2023 = (
    weather_2023
    .rename(
        columns={
            0:'station',
            1:'date',
            2:'metric',
            3:'temperature' 
        }
    ).drop(
        columns={
            [4, 5, 6, 7],
            errors='ignore'
        }
    )
)

weather_2024 = (
    weather_2024
    .rename(
        columns={
            0:'station',
            1:'date',
            2:'metric',
            3:'temperature' 
        }
    ).drop(
        columns={
            [4, 5, 6, 7],
            errors='ignore'
        }
    )
)

In [ ]:
# Join dfs and convert temperature
joined = pd.concat(
    [weather_2023, weather_2024],
    ignore_index=True
)

joined['temperature'] = joined['temperature'] / 10

In [ ]:
# Filter out Germany stations
joined = joined[joined['station'].str.startswith("GM")] 

# Join weather data and German stations
joined_2 = (
    pd.merge(
        joined,
        stations,
        on='station',
        how='left'
    )
)

joined_2.head() joined_2['variable'] = 'Regional Weather - ' + joined_2['metric']
joined_2 = joined_2.drop(
    columns=[
        'station', 'metric'
        ]
    )

joined_2 = joined_2.drop_duplicates(
    subset=[
        'date', 'state', 'variable'
        ]
    )joined_2.rename(
    columns={
        'temperature':'value'
        },inplace=True
)

joined_2.head() 

In [ ]:
# Parse dates in YYYYMMDD format into datetime objects (e.g, 20240601 -> 2024-06-01),
# then convert them to a formatted string representation, e.g., 01-Jun-24
joined_2['date'] = pd.to_datetime(joined_2['date'], format='%Y%m%d')
joined['date'] = joined_2['date'].dt.strftime('%d-%b-%y')

joined_2 # Concat dfs
final = pd.concat(
    [raw_2025, raw_2026, joined_2]
)

# ---------------- Parse dates into W/e Sunday format ----------------
# 1. Convert the 'date' column to datetime, allowing mixed input formats.
# format='mixed' lets pandas infer the format for each row individually,
# while dayfirst=True ensures European-style dates(DD/MM/YYYY) are parsed correctly.
final['date'] = pd.to_datetime(
    final['date'], 
    format='mixed',
    dayfirst=True
    )

# 2. Normalise each date to the corresponding week-ending Sunday.
# dt.weekday returns the day number (Monday=0, ..., Sunday=6).
# (6 - weekday) calculates how many days to add to reach Sunday,
# and pd.to_timedelta shifts the date forward by that many days.
final['date'] = final['date'] + pd.to_timedelta(6 - final['date'].dt.weekday, unit='D')

# Set 'ppg' and 'kpi' columns to NaN (since no values are available)
final['kpi'] = np.nan
final['ppg'] = np.nan

# Remove duplicates
final = final.drop_duplicates(
    subset=[
        'date', 'state', 'variable', 'kpi', 'ppg'
    ]
) 

df_export = final.copy()
df_export['date'] = df_export['date'].dt.strftime('%d-%b-%y')


In [ ]:
df_export.to_csv(
    r"C:\(...)\DE 2023-2026Q1 State Level Weather Upload.csv",
    index=False
)